# PE6201 A2 - Problem B (Final) - guided tour

Follows the scaffold's own principle stated in its README: **notebooks explore, modules ship.** Every cell below imports from `src/` and `experiments/` - there is no decision-making logic written in this notebook. If you are looking for the actual agent loop, tool layer, or guardrails, read `src/agent.py`, `src/tools.py`, `src/guardrails.py` directly; this notebook only calls them and prints what happens.

Run top to bottom. No API key, no network, no package installs - everything here uses the free, deterministic scripted backend.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join("..", "src"))
import config
print(config.summary())

## 1. Generate and validate the extended data (once)

The shipped 15 referrals plus this project's 20 new ones. Run the checker - it holds fingerprints of every shipped row, so if this prints any `FAIL` lines, something shipped was accidentally edited.

In [ ]:
!python3 ../data/make_fixtures_B_final.py
!python3 ../data/check_my_data_final.py

## 2. Walk one case turn by turn: REF-5602 (the brief's worked booking)

`agent.run_case` returns the full decision record: every tool call, every turn, tokens, cost, guardrail events, and the final decision.

In [ ]:
from agent import run_case
record = run_case("REF-5602", verbose=True)
print()
print(json.dumps({k: record[k] for k in ("decision", "booked", "reason", "turns", "tool_call_count", "stopped_by")}, indent=2))

**The trap this case is built to catch:** three OPH slots dated *inside* this referral's 8-week window are free and earlier than the slot actually booked - but they are in the **urgent** band, not routine. Only because `band` is a required, filtered argument on `get_clinic_slots` (see `tools.py`) does the run correctly skip them and book the first *routine* slot with capacity, five weeks out.

In [ ]:
import tools
window = tools.compute_window(8)
print("window:", window)
print("all OPH slots inside that window, any band, for comparison:")
for s in tools._load("clinic_slots"):
    if s["specialty"] == "OPH" and window["from"] <= s["date"] <= window["to"]:
        print(" ", s)

## 3. The two other regression anchors

In [ ]:
for cid in ("REF-5614", "REF-5590"):
    r = run_case(cid)
    print(cid, "->", r["decision"], "|", r.get("trigger") or r.get("missing"), "|", r["turns"], "turns")

## 4. The full 35-case evaluation set (D4/D5a)

In [ ]:
!cd ../src && python3 run_eval.py

## 5. D2(c) - sequential vs parallel, D3(b) - guardrail checklist, D6 - cost model, D7 - two failures

Each of these is its own module under `experiments/`, with saved evidence under `results/`. Run them here or from the shell - identical output either way.

In [ ]:
!python3 ../experiments/d2c_parallelism/run_comparison.py

In [ ]:
!python3 ../experiments/d3_guardrails/run_guardrail_cases.py

In [ ]:
!python3 ../experiments/d6_cost/run_cost_model.py

In [ ]:
!python3 ../experiments/d7_failures/failure_1_loop.py

In [ ]:
!python3 ../experiments/d7_failures/failure_2_slot_interface.py

## 6. What is NOT run here, and why

`experiments/d2b_descriptors/run_live_comparison.py` and `experiments/d5_models/run_live_battery.py` both need `config.BACKEND = "live"` and an API key (`OPENROUTER_API_KEY`). Neither is available in the environment this project was authored in - see `../STATUS.md`. Both scripts are complete and will run as soon as a key is exported; they fail loudly with instructions rather than silently producing nothing if you try them without one.